# 04. Data Retrieval & Hybrid Multi-Agent Orchestration

Covers **Attributes 10, 11, 12, 23**:
- Structured SQL fact retrieval
- Unstructured BM25 + metadata document retrieval
- Inline document citations ([DOC-xxx])
- Hybrid multi-source orchestration
- Metadata, tag, and recency document filtering

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1] if "high_level" in str(pathlib.Path.cwd()) or "capabilities" in str(pathlib.Path.cwd()) else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from src.orchestrator import Orchestrator
from src.llm_client import MockLLMClient
from src.tools.sql_tool import run_query, validate_sql
from src.tools.retrieval_tool import get_index
from src.tools.code_tool import run_code
from src.formatting import format_value, rows_to_markdown_table

print("AB InBev Enterprise Q&A Agent Pipeline Loaded.")

AB InBev Enterprise Q&A Agent Pipeline Loaded.

### 1. Document Retrieval with BM25 & Metadata Filtering

In [2]:
idx = get_index()
docs = idx.search("Corona Cero Olympic Games Paris sponsorship", k=3)
print(f"Retrieved {len(docs)} documents:")
for d in docs:
    print(f"- [{d.doc_id}] {d.title} (Date: {d.date}, Score: {d.score})")
    print(f"  Tags: {d.tags} | Brands: {d.brands}")
    print(f"  Excerpt: {d.excerpt}\n")

Retrieved 3 documents:
- [DOC-001] AB InBev Announces Corona Cero as Worldwide Olympic Partner (Date: 2025-01-15, Score: 11.75)
  Tags: ['olympics', 'non_alcoholic', 'beyond_beer', 'global_partnership'] | Brands: ['Corona Cero', 'Corona']
  Excerpt: # AB InBev Announces Corona Cero as Worldwide Olympic Partner *2025-01-15 — Press Release* Anheuser-Busch InBev (AB InBev) today announced a historic global partnership with the International Olympic Committee (IOC), naming Corona Cero the Worldwide Olympic Partner for the Olympic and Paralympic Games. This partnership highlights AB InBev's commitment to responsible consumption and accelerating the growth of non-alcoholic beer. Corona...

- [DOC-002] Q1 2025 Earnings Commentary: Corona Cero Acceleration in Europe and Mexico (Date: 2025-04-30, Score: 7.178)
  Tags: ['earnings', 'non_alcoholic', 'beyond_beer', 'growth'] | Brands: ['Corona Cero']
  Excerpt: # Q1 2025 Earnings Commentary: Corona Cero Acceleration in Europe and Mexico *2025-04-3

### 2. Pure Metadata Filtering (No Search Keywords)

In [3]:
docs_sustainability = idx.search("", k=4, tags=["sustainability"])
print(f"Sustainability updates retrieved: {len(docs_sustainability)}")
for d in docs_sustainability:
    print(f"- [{d.doc_id}] {d.title} ({d.date})")

Sustainability updates retrieved: 4
- [DOC-014] Strategy Memo: Circular Packaging and Net-Zero Carbon Brewery Roadmap (2025-09-14)
- [DOC-022] Returnable Glass Bottle Initiative Expands Across Latin America (2025-06-25)
- [DOC-013] Sustainability Update: Watershed Protection and Water Stewardship in Mexico (2025-05-20)
- [DOC-012] AB InBev Reaches 100% Renewable Electricity at Key Breweries in Belgium and Brazil (2025-02-25)

### 3. Hybrid Orchestration (Structured SQL + Unstructured Documents)

In [4]:
orch = Orchestrator(llm_router=MockLLMClient(), llm_worker=MockLLMClient())
q = "Why did Corona Cero grow in the United Kingdom, any press releases?"
r = orch.handle_turn(q)
print(f"User: {q}")
print(f"Sub-Agents Used: {r.sub_agents_used}")
print(f"Citations: {r.citations}")
print(f"\nAnswer:\n{r.answer}")

User: Why did Corona Cero grow in the United Kingdom, any press releases?
Sub-Agents Used: ['structured', 'unstructured']
Citations: [{'doc_id': 'DOC-001', 'title': 'AB InBev Announces Corona Cero as Worldwide Olympic Partner', 'date': '2025-01-15', 'source_type': 'press_release'}, {'doc_id': 'DOC-002', 'title': 'Q1 2025 Earnings Commentary: Corona Cero Acceleration in Europe and Mexico', 'date': '2025-04-30', 'source_type': 'earnings_commentary'}, {'doc_id': 'DOC-003', 'title': 'Market Research Note: The Moderation Mega-Trend and No/Low Alcohol Beer', 'date': '2025-06-20', 'source_type': 'market_research'}, {'doc_id': 'DOC-028', 'title': 'Annual Corporate Summary: Full-Year Operational and Financial Highlights', 'date': '2025-12-10', 'source_type': 'earnings_commentary'}, {'doc_id': 'DOC-019', 'title': 'Competitive Intelligence: Heineken NV Expansion in European Premium and 0.0%', 'date': '2025-03-14', 'source_type': 'competitor_intel'}]

Answer:
In 2025, Corona in United Kingdom reco